# How it works: Phases 0–3

*When Adaptive Posteriors Become Confidently Wrong*

A companion to `phases_0_3_walkthrough.ipynb`. That notebook shows **what the results
are**; this one shows **how the method works and why it fails**, on toy-sized runs you can
watch execute. The argument in four steps:

1. **Validate the likelihood assumption** — is the product likelihood legitimate when the
   design is adaptive?
2. **Show the implementation matches Rotem** — for five of six designs, and exactly why
   the sixth cannot.
3. **Show calibration holds when the model is right** — the control condition.
4. **Show where it fails** — the tail — and explain the mechanism.

> ### ⚠️ These numbers are illustrations, not results
> Every simulation here uses 30–150 replicates, 2–3k particles and the fast particle
> posterior, so cells finish in seconds. The published results use 250–300 replicates,
> 10k particles and the reference grid posterior. **Numbers printed here must never be
> quoted.** They are sized to make a mechanism visible, not to measure it. The real
> numbers live in `results/summaries/` and their status is in `docs/claims.md`.

**The setting.** A sequential sensitivity experiment. At each step we choose a stimulus
level $x$, observe a binary response $y \in \{0,1\}$, and update a posterior over a probit
response curve $p_\theta(x) = \Phi((x-\mu)/\sigma)$. We want credible intervals for
response quantiles $q_p$ — the stimulus at which the response probability is $p$. The
upper tail, $q_{0.95}$ and $q_{0.99}$, is where the decisions get made.

In [ ]:
import json
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import lognorm, norm

REPO = pathlib.Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from src.calibration import uniformity_report
from src.priors import rotem_prior, rotem_stimulus_grid
from src.response_models import ProbitCurve, binary_entropy
from src.simulator import ExperimentConfig, build_curve, run_experiment, true_targets

PRIOR = rotem_prior("well", "independent")      # mu0 = 30, the truth
SPEC = PRIOR.spec
PROBIT = {"family": "probit", "mu": 30.0, "sigma": 3.0}
TAIL = {"family": "tail_perturbed", "shift95": 1.0}

plt.rcParams["figure.figsize"] = (8, 3.4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 130)


def toy_config(policy, curve, n_steps=30, spec=SPEC, n_particles=2000, grid_points=80):
    """Small, fast version of the real experiment configuration."""
    return ExperimentConfig(
        policy=policy, prior=spec.config(), curve=curve, calibration="well",
        n_steps=n_steps, slices=(n_steps,),
        targets=("mu", "sigma", "q0.5", "q0.95", "q0.99"),
        n_particles=n_particles, grid_points=grid_points, grid_half_width=14.0,
    )


print("prior: mu ~ N(%.0f, %.0f^2),  sigma lognormal(%.3f, %.3f)"
      % (SPEC.mu0, SPEC.sigma_mu, SPEC.sigma_log_location, SPEC.tau_scale))
print("stimulus grid: [%.0f, %.0f]" % (SPEC.mu0 - 14, SPEC.mu0 + 14))

---
# 1. Validate the likelihood assumption

**The worry.** In an adaptive experiment, $x_2$ is chosen *using* $y_1$, $x_3$ using
$y_1,y_2$, and so on. The observations are therefore not independent. So is

$$L(\theta) = \prod_i p_\theta(y_i \mid x_i)$$

— which treats them as if they were — even the right likelihood? If not, every posterior
in this project is built on sand.

Let's first watch one experiment actually happen.

In [ ]:
run = run_experiment(toy_config("entropy_vector", PROBIT), replicate=0, seed_base=1)

steps = pd.DataFrame({"step": np.arange(1, 13),
                      "stimulus x": run["x"][:12].round(2),
                      "response y": run["y"][:12]})
print("The first 12 steps of one adaptive experiment:")
print(steps.to_string(index=False))
print("\nEach x depends on all previous y's -- that is what makes the design adaptive,")
print("and what makes the independence question worth asking.")

**The resolution.** Write the probability of the whole history $H_n$:

$$p_\theta(H_n) = \prod_i \underbrace{\pi(x_i \mid H_{i-1})}_{\text{the design's choice}} \times \prod_i \underbrace{p_\theta(y_i \mid x_i)}_{\text{the response model}}$$

The design term depends on the *past data*, but if the policy never looks at the true
$\theta$ — it only ever sees data, which is what "ignorable" means — then that term
contains no $\theta$. It is a constant with respect to $\theta$, so it cancels in the
posterior. The product likelihood is **exact**, not an approximation.

**How to check it.** Compute $\log p_\theta(H_n) - \sum_i \log p_\theta(y_i\mid x_i)$ on a
grid of $\theta$ values. If the design term is free of $\theta$, this difference is a
**flat line**.

And a check that can only pass is worthless, so we also run a deliberately *cheating*
policy — an oracle that peeks at the true $\theta$ when choosing $x$. That one must break.

In [ ]:
from src.factorization import (factorisation_residual, posterior_from_loglik,
                               product_loglik, total_variation)
from src.policies import EntropyVectorPolicy, OraclePolicy, PolicyState
from src.rotem_particles import ParticlePosterior

CAND = rotem_stimulus_grid("well", n_points=60)
CURVE = ProbitCurve(30.0, 3.0)
NP, NS = 800, 12
MU, SG = np.meshgrid(np.linspace(26, 34, 9), np.linspace(1.5, 5.0, 8), indexing="ij")
MU, SG = MU.ravel(), SG.ravel()


def simulate(policy, seed):
    rng = np.random.default_rng(seed)
    post = ParticlePosterior(PRIOR, NP, rng)
    state = PolicyState(candidates=CAND, posterior=post)
    xs, ys = [], []
    u = np.random.default_rng(seed + 5000).random(NS)
    for k in range(NS):
        x, _ = policy.select(state, rng)
        y = int(u[k] < float(np.atleast_1d(CURVE.prob(np.array([x])))[0]))
        post.update(x, y)
        xs.append(x); ys.append(y)
        state.x_hist, state.y_hist, state.step = xs, ys, k + 1
        state.invalidate()
    return np.asarray(xs), np.asarray(ys)


def residual(factory, seed):
    x, y = simulate(factory(), seed)
    r = factorisation_residual(factory(), PRIOR, CAND, x, y, MU, SG,
                               n_particles=NP, seed=seed)
    lp, prod = PRIOR.logpdf(MU, SG), product_loglik(x, y, MU, SG)
    tv = total_variation(posterior_from_loglik(prod, lp),
                         posterior_from_loglik(prod + r, lp))
    return r, tv


r_ok, tv_ok = residual(lambda: EntropyVectorPolicy(), 11)
r_bad, tv_bad = residual(lambda: OraclePolicy(CURVE, p=0.5, sharpness=2.0), 37)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, r, ttl in [(axes[0], r_ok, "ignorable (entropy_vector)"),
                   (axes[1], r_bad, "NON-ignorable (oracle)")]:
    ax.plot(r[np.isfinite(r)], ".-", ms=4, lw=0.8)
    ax.set(xlabel="grid point (72 values of $\\theta$)", title=ttl)
axes[0].set_ylabel("design-term residual")
axes[0].set_ylim(r_ok.mean() - 1, r_ok.mean() + 1)
plt.tight_layout(); plt.show()

print("ignorable      : spread %.2e nats, posterior total variation %.2e"
      % (np.ptp(r_ok[np.isfinite(r_ok)]), tv_ok))
print("non-ignorable  : spread %.2f nats, posterior total variation %.3f"
      % (np.ptp(r_bad[np.isfinite(r_bad)]), tv_bad))

**Left panel: a flat line.** The residual is constant to ~$10^{-16}$, and the two
posteriors are identical to within $10^{-16}$ total variation. Adaptive dependence costs
us nothing — the simple product likelihood *is* the exact likelihood.

**Right panel: not flat.** The oracle's design term varies by ~84 nats across $\theta$,
and the two posteriors differ by 0.77 in total variation. This is the guard rail: it
proves the left panel is a real check and not an implementation that quietly ignores the
design term.

**Why this matters for the whole thesis.** It closes off the easy explanation. When we
find broken intervals in Part 4, nobody can say "well, you used the wrong likelihood."
The likelihood is exact; the failure has to come from somewhere else.

---
# 2. Does the implementation match Rotem's?

Before stress-testing a method, you must be running *her* method. Six designs are
compared against her published tables. Five agree; `entropy_sigma` does not, in any
setting.

Start by just watching what each design does with its stimuli.

In [ ]:
designs = ["bruceton", "dror_steinberg", "entropy_mu", "entropy_vector", "entropy_sigma"]
runs = {d: run_experiment(toy_config(d, PROBIT), replicate=0, seed_base=5) for d in designs}

fig, axes = plt.subplots(1, 5, figsize=(12, 2.9), sharey=True)
for ax, d in zip(axes, designs):
    r = runs[d]
    hit = r["y"] == 1
    ax.scatter(np.arange(len(r["x"]))[hit], r["x"][hit], s=16, marker="^", label="y=1")
    ax.scatter(np.arange(len(r["x"]))[~hit], r["x"][~hit], s=16, marker="v", label="y=0")
    ax.axhline(30.0, color="k", ls="--", lw=1)
    ax.axhspan(43.6, 44.4, color="crimson", alpha=0.15)
    ax.axhspan(15.6, 16.4, color="crimson", alpha=0.15)
    ax.set(title=d, xlabel="step", ylim=(15, 45))
axes[0].set_ylabel("stimulus $x$")
axes[0].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

print("red bands = the edges of the candidate grid;  dashed line = the true median (30)")
print()
for d in designs:
    s = runs[d]["slices"][30]
    print("%-16s distinct levels %4.0f   overlapping pattern: %-5s  posterior sd(mu) %.2f"
          % (d, s["n_distinct_x"], bool(s["overlap"]), s["mu_sd"]))

**Four designs cluster around the median; `entropy_sigma` sits on the two grid edges and
nothing else.** It produces no *overlapping pattern* — no interleaving of 0s and 1s that
would pin down where the curve crosses — and its posterior for $\mu$ stays wide.

That looks like a bug. It isn't. Here is the criterion it maximises, computed in closed
form.

In [ ]:
# At the prior, with mu ~ N(mu0, s_mu^2) independent of sigma, the response probability
# marginalised over mu is Phi((x - mu0) / sqrt(sigma^2 + s_mu^2)) -- so I(sigma; Y | x)
# reduces to a one-dimensional integral we can evaluate to machine precision.
x_grid = np.linspace(16, 44, 281)
u = (np.arange(4000) + 0.5) / 4000
sig_nodes = lognorm.ppf(u, s=SPEC.tau_scale, scale=np.exp(SPEC.sigma_log_location))

p_sigma = norm.cdf((x_grid[:, None] - SPEC.mu0) / np.sqrt(sig_nodes[None, :] ** 2 + SPEC.sigma_mu ** 2))
I_sigma = binary_entropy(p_sigma.mean(1)) - binary_entropy(p_sigma).mean(1)

rng = np.random.default_rng(0)                       # joint prior draws for the contrast
mu_d, sg_d = PRIOR.sample(4000, rng)
p_joint = norm.cdf((x_grid[:, None] - mu_d[None, :]) / sg_d[None, :])
I_joint = binary_entropy(p_joint.mean(1)) - binary_entropy(p_joint).mean(1)

fig, ax = plt.subplots()
ax.plot(x_grid, I_sigma / I_sigma.max(), lw=2, label=r"$I(\sigma;Y\mid x)$  — entropy_sigma")
ax.plot(x_grid, I_joint / I_joint.max(), lw=2, ls="--",
        label=r"$I((\mu,\sigma);Y\mid x)$  — entropy_vector")
ax.axvline(SPEC.mu0, color="k", ls=":", lw=1)
ax.annotate("$x=\\mu_0$: criterion is\nexactly ZERO", xy=(30, 0.02), xytext=(31.5, 0.35),
            fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.8))
ax.set(xlabel="stimulus $x$", ylabel="criterion (scaled to its own max)",
       title="Why entropy_sigma goes to the edges")
ax.legend(fontsize=8)
plt.show()

print("I(sigma;Y) at x = mu0 : %.2e   (exactly zero)" % I_sigma[np.argmin(abs(x_grid - 30))])
print("I(sigma;Y) at the edges: %.6f / %.6f  (tied)" % (I_sigma[0], I_sigma[-1]))

**This is the explanation, and it is a one-liner.** At $x=\mu_0$ the response probability
is $\Phi(0)=1/2$ **whatever $\sigma$ is**. A coin flip you'd get regardless of the answer
teaches you nothing, so the information about $\sigma$ is *exactly zero* at the centre.
It grows the further out you go, and is maximised at whatever the grid edges happen to be.

Compare the dashed line: the *joint* criterion peaks near the median, because that is
where you learn about $\mu$. Adding $\mu$ to the objective is what keeps the design sane.

So `entropy_sigma` is behaving **correctly**. Our implementation was audited against this
exact reference: correlation **0.996**, and it picks a stimulus whose exact criterion
value is **1.000** of the optimum (`results/summaries/phase1_quadrature_audit.csv`).
The design is doing precisely what "maximise information about $\sigma$ alone" asks.

Now the consequence — and why the resulting MSEs look bizarre.

**The criterion above is a curve in $x$ — but the thing being learned is a *point in a
plane*.** `entropy_vector` targets $\theta=(\mu,\sigma)$, so the honest picture has
$\mu$ on one axis and $\sigma$ on the other. Here is the same mechanism drawn there.


In [ ]:
# The (mu, sigma) plane: what a single observation at x tells you about *where* theta is.
mu_ax, sg_ax = np.linspace(20, 40, 241), np.linspace(0.4, 9.0, 241)
MU, SG = np.meshgrid(mu_ax, sg_ax)
X_CENTRE, X_EDGE = SPEC.mu0, SPEC.mu0 + 14.0

u = (np.arange(4000) + 0.5) / 4000
sig_nodes = lognorm.ppf(u, s=SPEC.tau_scale, scale=np.exp(SPEC.sigma_log_location))
mu_d, sg_d = PRIOR.sample(4000, np.random.default_rng(0))

def _mis(xv):
    """Exact I(sigma;Y|x) and I((mu,sigma);Y|x) at the prior."""
    ps = norm.cdf((xv - SPEC.mu0) / np.sqrt(sig_nodes**2 + SPEC.sigma_mu**2))
    pj = norm.cdf((xv - mu_d) / sg_d)
    return (max(binary_entropy(ps.mean()) - binary_entropy(ps).mean(), 0.0),
            binary_entropy(pj.mean()) - binary_entropy(pj).mean())

fig, axes = plt.subplots(1, 3, figsize=(13.4, 4.0))
for ax, xv, tag, levs in [(axes[0], X_CENTRE, "centre", [.1, .25, .5, .75, .9]),
                          (axes[1], X_EDGE, "edge", [.90, .95, .98, .995])]:
    P = norm.cdf((xv - MU) / SG)
    I_s, I_j = _mis(xv)
    im = ax.pcolormesh(mu_ax, sg_ax, P, cmap="RdBu_r", vmin=0, vmax=1, shading="auto")
    cs = ax.contour(mu_ax, sg_ax, P, levels=levs, colors="k", linewidths=.7, alpha=.6)
    ax.clabel(cs, fontsize=6, fmt="%.2f")
    ax.scatter(mu_d[:700], sg_d[:700], s=1.5, c="k", alpha=.18, linewidths=0)
    ax.axvline(SPEC.mu0, color="k", ls=":", lw=1.3)
    ax.set(xlabel=r"$\mu$", xlim=(20, 40), ylim=(0.4, 9.0),
           title=r"$x=%.0f$ (%s):  $I(\sigma)=%.3f$,  $I(\mu,\sigma)=%.3f$"
                 % (xv, tag, I_s, I_j))
    fig.colorbar(im, ax=ax, label=r"$P(Y=1\mid\mu,\sigma,x)$")
axes[0].set_ylabel(r"$\sigma$")

bbox = dict(boxstyle="round,pad=0.35", fc="white", ec="0.4", lw=.7, alpha=.92)
axes[0].text(.03, .96, "along the dotted line\n" r"$P\equiv 1/2$ for every $\sigma$" "\n"
             r"$\Rightarrow$ zero info about $\sigma$",
             transform=axes[0].transAxes, va="top", fontsize=7.5, bbox=bbox)
axes[1].text(.03, .04, "along the dotted line $P$ varies,\n"
             r"but every $(\mu,\sigma)$ predicts $Y\approx 1$" "\n"
             r"$\Rightarrow$ little info about anything",
             transform=axes[1].transAxes, va="bottom", fontsize=7.5, bbox=bbox)

sg = np.linspace(0.4, 9.0, 400)
for xv, tag, ls in [(X_CENTRE, "centre", "-"), (X_EDGE, "edge", "--")]:
    axes[2].plot(sg, norm.cdf((xv - SPEC.mu0) / np.sqrt(sg**2 + SPEC.sigma_mu**2)),
                 ls, lw=2, label=r"$x=%.0f$ (%s)" % (xv, tag))
axes[2].set(xlabel=r"$\sigma$", ylabel=r"$P(Y=1\mid\sigma,x)$   (marginal over $\mu$)",
            ylim=(-0.03, 1.03), title=r"what $I(\sigma;Y\mid x)$ actually sees")
axes[2].legend(fontsize=8, loc="center left")
fig.suptitle(r"The vector target lives in the $(\mu,\sigma)$ plane "
             r"- the scalar-$\sigma$ criterion only sees the slice", y=1.03)
plt.tight_layout(); plt.show()

for xv, tag in [(X_CENTRE, "centre"), (X_EDGE, "edge")]:
    print("x=%-7s I(sigma;Y)=%.6f   I((mu,sigma);Y)=%.6f" % ((tag,) + _mis(xv)))


**Read the left panel up the dotted line.** At $x=\mu_0$ the colour never changes as
$\sigma$ grows — $P\equiv\tfrac12$ all the way up — so an observation there cannot
discriminate one $\sigma$ from another. That is the *geometric* reason
$I(\sigma;Y\mid x)$ is exactly zero at the centre, and the contours fanning out of
$(\mu_0, 0)$ show it is the only place where that happens.

But the same panel is where the *vector* criterion is largest ($0.403$ nats): moving
left-to-right across the dotted line changes the colour sharply, and that is information
about $\mu$.

**The right panel is the trap.** At the grid edge the $\sigma$-direction finally carries
signal — which is why `entropy_sigma` goes there — but the whole panel is red: every
$(\mu,\sigma)$ in the prior predicts $Y=1$. The observation is nearly deterministic, so
the *total* information collapses from $0.403$ to $0.066$ nats. `entropy_sigma` is
maximising a share of a vanishing quantity.


In [ ]:
R = 30
rows = []
for prior_label, spec in [("prior centred ON the truth", rotem_prior("well", "independent").spec),
                          ("prior centred 8 units OFF", rotem_prior("poor", "independent").spec)]:
    for pol in ["entropy_vector", "entropy_sigma"]:
        cfg = toy_config(pol, PROBIT, spec=spec)
        recs = [run_experiment(cfg, r, seed_base=21)["slices"][30] for r in range(R)]
        err = np.array([s["mu_mean"] - s["mu_true"] for s in recs])
        rows.append({"prior": prior_label, "design": pol,
                     "overlap rate": np.mean([s["overlap"] for s in recs]),
                     "posterior sd(mu)": np.mean([s["mu_sd"] for s in recs]),
                     "MSE(mu)": (err ** 2).mean()})

print("Toy, %d replicates per row -- illustration only:" % R)
pd.DataFrame(rows).round(3)

**The trap.** `entropy_sigma` never learns $\mu$, so its reported $\mu$ error is decided
entirely by *where the prior happens to sit*:

- Prior centred on the truth → the posterior stays at the prior, which is already right →
  MSE looks **spectacular**.
- Prior centred 8 units off → the posterior stays at the prior, which is wrong →
  MSE looks **terrible**.

In the full runs this shows up as MSE($\mu$) = 0.013 against Rotem's 2.061 in one setting
(160× "better") and 42.0 against her 1.805 in another (23× worse). You do not beat a
published result by 160× by accident — that asymmetry is the fingerprint of a design that
isn't learning.

**Why this can't be reconciled with her numbers.** Under the off-centre prior she reports
MSE($\mu$) = 1.805 for this design — an RMS error of **1.34** from a prior **8 units**
away, comparable to designs *built* to estimate $\mu$. No design maximising
$I(\sigma;Y\mid x)$ can do that, for the reason plotted above: it never samples where
$\mu$ is identified. So her published table cannot have been produced by the criterion her
own §5.3 specifies.

**What that does and does not settle.** It puts the inconsistency between her text and her
numbers, not in this implementation — which is verified exact against the closed form. It
does *not* say which of the two is at fault: her code may implement a different objective
than the text describes, or the tabulated column may not be the one it is labelled as.
Telling those apart needs her code, so the status stays **Unresolved**
(`docs/discrepancies.md` D13, claim C2b) rather than being written up as her error.

Here is the verdict for all six designs, from the full 250-replicate runs.

In [ ]:
vs_pub = pd.read_csv(REPO / "results" / "summaries" / "phase1_vs_published.csv")
verdict = (vs_pub.groupby("policy")
           .agg(cells=("within_2se", "size"), agreeing=("within_2se", "sum"),
                max_abs_z=("z_combined", lambda s: s.abs().max()))
           .sort_values("max_abs_z"))
print("REAL results (250 replicates/cell), agreement within 2 combined SE:")
verdict.round(2)

Five designs reproduce. `entropy_sigma` agrees in 2 of 28 cells with a max $|z|$ over
1000 — a structural disagreement, exactly as the mechanism above predicts, not a tuning
difference. It is excluded from everything downstream; Phases 2 and 3 use
`entropy_vector` and `uniform_grid` only.

---
# 3. Is it calibrated when the model is right?

This is the control condition. Before claiming a failure we must show the machinery works
when nothing is wrong — otherwise any later breakage might just be a bug.

**Simulation-based calibration (SBC).** Draw $\theta$ from the prior, simulate a full
adaptive experiment from *that* $\theta$, then ask where the truth falls in the posterior.
Repeat. If the posterior is honest, the truth is equally likely to land anywhere — the
**rank statistic is uniform**. This tests the entire pipeline end to end, with the adaptive
policy in the loop.

In [ ]:
from src.calibration import sbc_replicate

N_SBC = 150
cfg = toy_config("entropy_vector", PROBIT)
t0 = time.time()
draws = [sbc_replicate(cfg, r, seed_base=7, use_reference=False) for r in range(N_SBC)]
print("%d SBC draws in %.0fs" % (N_SBC, time.time() - t0))

fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
out = []
for ax, tgt in zip(axes, ["mu", "q0.95", "q0.99"]):
    ranks = np.array([d[f"{tgt}_rank_particle"] for d in draws])
    rep = uniformity_report(ranks, n_bins=10)
    # Coverage is read OFF THE RANK STATISTIC, not by forming an interval and
    # testing containment -- so the coverage and SBC layers agree by construction.
    covered = (ranks > 0.025) & (ranks < 0.975)
    ax.hist(ranks, bins=10, range=(0, 1), edgecolor="white")
    ax.axhline(N_SBC / 10, color="crimson", ls="--", lw=1)
    ax.set(title="%s   (KS p = %.2f)" % (tgt, rep["ks_p"]), xlabel="rank of the truth")
    out.append({"target": tgt, "ks_p": rep["ks_p"], "mean_rank": rep["mean"],
                "coverage": covered.mean()})
axes[0].set_ylabel("count")
plt.tight_layout(); plt.show()

print("flat histogram + large KS p-value = calibrated.  Toy scale, illustration only:")
pd.DataFrame(out).round(3)

**Flat histograms.** The truth lands uniformly across the posterior, including for the
tail quantiles, with the adaptive design running. Nothing is broken.

In the real run (600 draws, reference posterior) this passes on **every** target under
both an adaptive and a non-adaptive design — claim C3. That is what licenses reading the
next section as a genuine effect rather than a software defect.

---
# 4. Where it fails: the tail

Now make the model wrong — but only slightly, and only where it is hard to notice.

The **tail-perturbed curve** is *exactly* a probit below the 0.85 quantile, and bends away
only above it. It is not a strawman: it was constructed to defeat the mechanism found in
Phase 2, where a globally mis-shaped curve inflated the posterior enough to absorb its own
bias. Here the fitted family is right everywhere the design looks.

In [ ]:
probit_c, tail_c = build_curve(PROBIT), build_curve(TAIL)
xs = np.linspace(20, 46, 400)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(xs, probit_c.prob(xs), lw=2, label="fitted family (probit)")
axes[0].plot(xs, tail_c.prob(xs), lw=2, ls="--", label="truth (tail-perturbed)")
axes[0].axhline(0.85, color="k", ls=":", lw=1)
axes[0].annotate("identical below p=0.85", xy=(28, 0.4), fontsize=8)
axes[0].set(xlabel="stimulus $x$", ylabel="P(response)", title="The two curves")
axes[0].legend(fontsize=8, loc="lower right")

ps = np.array([0.5, 0.85, 0.95, 0.99])
tq = np.array([tail_c.quantile(p) for p in ps])
pq = np.array([probit_c.quantile(p) for p in ps])
axes[1].bar(np.arange(4) - 0.2, pq, 0.4, label="probit")
axes[1].bar(np.arange(4) + 0.2, tq, 0.4, label="tail-perturbed")
axes[1].set(xticks=range(4), xticklabels=[f"q{p}" for p in ps], ylabel="stimulus",
            title="Where they disagree", ylim=(25, 46))
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(pd.DataFrame({"p": ps, "probit": pq.round(2), "tail-perturbed": tq.round(2),
                    "gap": (tq - pq).round(2)}).to_string(index=False))

The two curves are **identical** at $q_{0.5}$ and $q_{0.85}$, then separate by 3.0 units
at $q_{0.95}$ and 5.9 at $q_{0.99}$.

Now the mechanism. Where does the adaptive design actually put its observations?

In [ ]:
cfg_tail = toy_config("entropy_vector", TAIL, n_steps=50)
sampled = np.concatenate([run_experiment(cfg_tail, r, seed_base=31)["x"] for r in range(30)])

fig, ax = plt.subplots()
ax.hist(sampled, bins=40, color="steelblue", edgecolor="white", label="where the design samples")
for p, c in [(0.95, "darkorange"), (0.99, "crimson")]:
    ax.axvline(tail_c.quantile(p), color=c, lw=2, label=f"true $q_{{{p}}}$ = {tail_c.quantile(p):.1f}")
ax.axvline(tail_c.quantile(0.85), color="k", ls=":", lw=1.5, label="p=0.85: model starts to bend")
ax.set(xlabel="stimulus $x$", ylabel="observations",
       title="The design rarely samples where the deviation is large")
ax.legend(fontsize=8)
plt.show()

# How much of the data lands where the model is actually wrong, and how wrong is it there?
probit_q = lambda p: 30.0 + 3.0 * norm.ppf(p)
tbl = [{"above true q_p": f"q{p}", "x": round(tail_c.quantile(p), 1),
        "% of observations": round(100 * (sampled > tail_c.quantile(p)).mean(), 2),
        "model error there (units)": round(tail_c.quantile(p) - probit_q(p), 2)}
       for p in [0.85, 0.90, 0.95, 0.99]]
pd.DataFrame(tbl)

**There is the mechanism, and the table is the precise version of it.** The design does
cross $p=0.85$ — about 23% of observations land above it — but that is where the curve has
only *just* started to bend, and the error there is still ~0. The deviation grows
quadratically, so the two quantities move in opposite directions:

| where | model error | share of the data |
|---|---|---|
| above $q_{0.85}$ | 0.00 units | 23% |
| above $q_{0.95}$ | 3.00 units | **1.8%** |
| above $q_{0.99}$ | 5.88 units | **0.07%** |

**Where the model is meaningfully wrong, there is essentially no data.** Not because the
design is badly built — the opposite. It concentrates near the median because that is where
a binary response is most informative about $(\mu,\sigma)$, which is exactly what made it
good in Part 2. But that region is one where the probit is *exactly right*.

So the data contain almost no evidence against the model, and the posterior concentrates:
a well-fitting region looks like strong information. The tail quantile is then an
**extrapolation** from that region, reported with an interval whose width was set by how
well the model fits *where we looked*, not by how far we are extrapolating beyond it.
Confident, and wrong.

Let's measure it, with the two controls that make it interpretable.

In [ ]:
R = 60
arms = [("A  tail + adaptive", "entropy_vector", TAIL),
        ("B  tail + fixed",    "fixed_design",   TAIL),
        ("C  probit + adaptive", "entropy_vector", PROBIT)]

t0, cov = time.time(), {}
for name, pol, curve in arms:
    cfg = toy_config(pol, curve, n_steps=50, n_particles=3000, grid_points=100)
    recs = [run_experiment(cfg, r, seed_base=11, crn_seed=99)["slices"][50] for r in range(R)]
    cov[name] = {t: np.mean([s[f"{t}_covered"] for s in recs]) for t in ["q0.5", "q0.95", "q0.99"]}
print("%d replicates per arm, %.0fs" % (R, time.time() - t0))

cov = pd.DataFrame(cov).T
fig, ax = plt.subplots()
cov.T.plot.bar(ax=ax, rot=0, width=0.78)
ax.axhline(0.95, color="k", ls="--", lw=1)
ax.set(ylim=(0, 1.08), ylabel="coverage of a 95% interval", xlabel="target",
       title="Toy version of the main result (illustration, not the published numbers)")
ax.legend(fontsize=8, loc="lower left")
plt.show()
cov.round(2)

**Read the controls.** At the median all three arms are near nominal — nothing looks
broken. In the tail:

- **C** (probit + adaptive) stays at nominal → adaptivity alone is harmless.
- **B** (tail + fixed) drops somewhat → misspecification alone costs something.
- **A** (tail + adaptive) drops furthest → the two together are worse than either.

Neither ingredient is sufficient on its own; the damage comes from the **interaction**.
The adaptive design concentrates sampling where the model is right, which is exactly what
prevents it from ever discovering that the tail is wrong.

The toy shows the *direction*. The confirmatory run — 300 replicates, CRN-paired,
reference posterior, pre-registered — puts arm A at **0.633** at $q_{0.95}$ and **0.433**
at $q_{0.99}$ against a nominal 0.95, while arm C holds 0.97. See
`phases_0_3_walkthrough.ipynb` or `results/summaries/phase3b_confirm_summary.csv`.

---
## 4b. The real numbers, and the plot that should worry you

> **Everything from here on is the published data**, read from `results/raw/` and
> `results/summaries/` — 300 replicates per cell, reference posterior, pre-registered.
> Unlike the toy cells above, **these numbers are quotable.**

The obvious hope is that this is a small-sample problem: collect more data and it goes
away. Let's check.

In [ ]:
RAW, SUM = REPO / "results" / "raw", REPO / "results" / "summaries"
ARM = {"A": "A  tail + adaptive", "B": "B  tail + fixed", "C": "C  probit + adaptive"}

cf = pd.read_parquet(RAW / "phase3b_confirm_raw.parquet")

# The reference posterior was only computed at n=50, so the horizon uses the particle
# backend.  Check the two agree where both exist, otherwise this comparison is not safe:
chk = cf[cf.n == 50].groupby("label").agg(particle=("q0.99_covered", "mean"),
                                          reference=("q0.99_ref_covered", "mean"))
print("agreement at n=50 (justifies using the particle backend for n=20, 30):")
print(chk.round(3).to_string(), "\n")

hz = (cf.groupby(["label", "n"])
        .agg(cov99=("q0.99_covered", "mean"), width=("q0.99_width", "mean"),
             reps=("q0.99_covered", "size")).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for lab, g in hz.groupby("label"):
    axes[0].plot(g.n, g.cov99, "o-", lw=2, label=ARM[lab])
    axes[1].plot(g.n, g.width, "o-", lw=2, label=ARM[lab])
axes[0].axhline(0.95, color="k", ls="--", lw=1)
axes[0].set(xlabel="observations collected", ylabel="coverage of $q_{0.99}$",
            ylim=(0, 1.05), xticks=[20, 30, 50], title="More data, WORSE coverage")
axes[1].set(xlabel="observations collected", ylabel="mean interval width",
            xticks=[20, 30, 50], title="...while the interval keeps shrinking")
axes[0].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

hz.pivot(index="label", columns="n", values=["cov99", "width"]).round(2)

**More data makes it worse.** Arm A's coverage of $q_{0.99}$ falls **0.78 → 0.66 → 0.43**
as the experiment grows from 20 to 50 observations, while its interval *halves*,
15.7 → 8.0. Confidence rises as accuracy falls.

This is the opposite of how estimation is supposed to behave, and it rules out the
comfortable reading. It is not a small-sample problem that more data fixes — more data is
what *drives* it. Each extra observation lands near the median (Part 4), tightening the
posterior on a model that is wrong out in the tail, so the interval contracts around the
wrong value faster than the value corrects.

The control settles it: arm **C**, correct model, same design, same horizons, is flat at
0.95 → 0.95 → 0.97 while its interval shrinks the same way. Shrinking intervals are not
the problem. Shrinking intervals *around an unfalsifiable extrapolation* are.

In [ ]:
w_star = json.load(open(REPO / "configs" / "phase3_thresholds.json"))["thresholds"]["q0.99|n50"]
d50 = cf[cf.n == 50]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True, sharey=True)
for ax, lab in zip(axes, ["A", "C"]):
    g = d50[d50.label == lab]
    err = (g["q0.99_ref_mean"] - g["q0.99_true"]).abs()
    wid, cov = g["q0.99_ref_width"], g["q0.99_ref_covered"].astype(bool)
    ax.scatter(wid[cov], err[cov], s=13, alpha=0.45, label="interval covered the truth")
    ax.scatter(wid[~cov], err[~cov], s=13, alpha=0.75, color="crimson", label="MISSED")
    ax.axvline(w_star, color="k", ls="--", lw=1)
    ax.axvspan(0, w_star, color="crimson", alpha=0.06)
    fc = ((wid <= w_star) & (~cov)).mean()
    ax.set(xlabel="interval width  (narrow $\\rightarrow$)",
           title="%s\nnarrow AND wrong: %.0f%% of runs" % (ARM[lab], 100 * fc))
axes[0].set_ylabel("|error| in $q_{0.99}$")
axes[0].legend(fontsize=8, loc="upper right")
axes[0].annotate("false certainty:\nconfident and wrong", xy=(4.5, 9), fontsize=8,
                 color="crimson", ha="center")
plt.tight_layout(); plt.show()

print("shaded = narrower than w* = %.2f, the 25th percentile of width under the "
      "correctly-specified control" % w_star)

**What "false certainty" looks like.** Each dot is one experiment: how wide the interval
was, against how far the estimate actually was from the truth. A well-behaved method fills
the upper-right (wide when wrong) and the lower-left (narrow when right).

Arm A fills the **lower-right instead** — the shaded region, narrow *and* missing, on 25%
of runs. Arm C, correct model, leaves it nearly empty at 1%.

This is why coverage alone understates the problem. A wide interval that misses is honest:
it announced its own uncertainty. A narrow interval that misses gives the user a precise,
confident, wrong answer with nothing in the output to suggest doubt. $w^\ast$ is not
tuned to flatter the result — it is fixed at the 25th percentile of width under the
*correctly-specified* control, so "narrow" means narrow by the standard of a run where
nothing was wrong.

---
# 5. A different failure: the approximation, not the model

Everything above is about the *model* being wrong. There is a second, unrelated way to get
bad intervals: the **posterior itself** being computed badly. Rotem's posterior is a
weighted particle approximation; the reference is a refined grid. Keeping these two error
sources apart is a standing rule of the project — a computational defect must never be
reported as evidence about misspecification.

A natural guess is that the particle posterior struggles when the data have **no
overlapping pattern** (no interleaving of 0s and 1s to pin down where the curve crosses).
The no-overlap study has the power to test it: 3,304 runs across both arms.

In [ ]:
st = pd.read_csv(SUM / "phase3a_stratified.csv")
g99 = (st[st.target == "q0.99"].groupby(["arm", "sigma_true"])
       .agg(gap=("cov_diff", "mean"), cov_particle=("cov_particle", "mean"),
            cov_reference=("cov_reference", "mean"), runs=("n", "sum")).reset_index())

print("Marginally, it looks like an overlap effect:")
print(st[st.target == "q0.99"].groupby("arm").cov_diff.mean().round(3).to_string())
print("\nBut split by sigma:")

fig, ax = plt.subplots()
for arm, g in g99.groupby("arm"):
    ax.plot(g.sigma_true, g.gap, "o-", lw=2, ms=4, label=arm)
    for _, r in g.iterrows():                       # marker area ~ number of runs
        ax.scatter(r.sigma_true, r.gap, s=max(np.sqrt(r.runs) * 3, 12), alpha=0.35)
        ax.annotate("n=%d" % r.runs, (r.sigma_true, r.gap), textcoords="offset points",
                    xytext=(6, 6), fontsize=7)
ax.axhline(0, color="k", lw=1)
ax.set(xlabel=r"true $\sigma$ (steepness of the curve)",
       ylabel="particle $-$ reference coverage, $q_{0.99}$",
       title=r"The gap is a $\sigma$ effect, not an overlap effect")
ax.legend(fontsize=8, loc="lower right")
plt.show()

g99.round(3)

**The guess is wrong, and the way it is wrong is instructive.**

Marginally the numbers look like a clean overlap story: the particle posterior trails the
reference by **0.446** without an overlapping pattern and only **0.284** with one. Stratify
by $\sigma$ and it dissolves:

| $\sigma$ | no_overlap | overlap_control |
|---|---|---|
| 0.3 | −0.634 | −0.365 |
| 1.2 | −0.006 | 0.000 |

At $\sigma = 1.2$ the gap is **zero in both arms**. At $\sigma = 0.3$ it is large in **both**.
Overlap is not the driver — it is a *proxy* for a steep curve, because steep curves are
what fail to produce an overlapping pattern. Conditioning on overlap silently conditions
on $\sigma$, and the plot would have credited overlap with an effect that belongs to
steepness.

**Caveat, since the plot cannot show it honestly otherwise:** the `overlap_control` point
at $\sigma=1.2$ rests on **5 runs** (marker sizes are scaled by run count for this reason).
It is consistent with the others but carries no weight on its own. The load-bearing
comparison is at $\sigma=0.3$, where both arms have 1,000+ runs.

So the computational failure is a **small-$\sigma$ corner**, not an overlap phenomenon —
about 1.6% of runs unconditionally, and zero for $\sigma \ge 1.2$. It has a concrete fix
(trigger rejuvenation on ESS, or drop the KL threshold to 0.02) and it is a separate
chapter from everything in Parts 1–4. That separation is the point: had we plotted this
against overlap, it would read as a statement about experimental design rather than about
a resampling rule.

---
# Why this matters

**The chain.** The likelihood is exact (1) → the implementation is Rotem's (2) → the
machinery is calibrated when the model is right (3) → so the tail failure (4) is a real
property of the method, not an artefact of any of the three. And it is not the
approximation either (5), which fails in a different place for a different reason.

**The uncomfortable part.** Every diagnostic available *inside* the experiment looks
healthy. The model fits the observed data. The posterior is tight and getting tighter. The
median is recovered accurately. Nothing signals a problem — and the interval for
$q_{0.99}$ is wrong more often than it is right.

Three findings compound, and the order matters:

1. **It is invisible.** The design samples where the model is right, so the data carry
   almost no evidence against it.
2. **It is confident.** 25% of runs report an interval narrower than the correctly-specified
   control typically produces, *and* miss — against 1% when the model is right.
3. **It gets worse with more data.** Coverage falls 0.78 → 0.43 as the experiment grows,
   while the interval halves. Collecting more is not the remedy; it is the mechanism.

**Why it is worth a thesis.** These designs are used precisely where the upper tail is the
point — the dose, stress or exposure at which something fails 1% of the time. The
efficiency that makes adaptive design attractive is the same property that makes its tail
extrapolations untrustworthy, the failure is invisible from inside the experiment, and the
usual instinct of running longer makes it worse. That motivates the phases not yet built:
a theory of when it happens (Phase 4), a diagnostic computable online from the data at hand
(Phase 5), and a safeguard (Phase 6) — which is genuine research, since the exploratory
control tested here does **not** restore coverage (claim C10).

**Scope, stated plainly.** The confirmatory result is one constructed curve family at one
horizon; the screening atlas suggests it generalises — `robit` shows the same signature at
about half the size — but that is 30-replicate precision, not proof. Every result here
also predates the provenance contract: the manifests are unpinned and carry no `run_id`
(`D19`, `D20`), so the numbers are checkable against the files but not yet tied to a state
of the source. Current status for every claim is `docs/claims.md`.